In [39]:
# https://portal.inmet.gov.br/dadoshistoricos

O INMET possui estações automáticas e convencionais na capital. As principais que você deve buscar no banco de dados (BDMET) são:

- A602 Rio de Janeiro (Marambaia) Estação convencional (importante se houver falhas nas automáticas).

- A621	Rio de Janeiro - Vila Militar	Fundamental para captar as temperaturas mais altas da cidade (Zona Norte/Oeste).

- A636	Rio de Janeiro - Jacarepaguá	Representa bem a Zona Oeste e áreas de expansão urbana.

- A652	Rio de Janeiro - Forte de Copacabana	Representa a influência marítima e a Zona Sul.



Colunas:
1. Precipitação e Radiação (Energia e Massa)
PRECIPITAÇÃO TOTAL, HORÁRIO (mm): Volume de chuva acumulado no intervalo de uma hora. É uma variável acumulada.

RADIACAO GLOBAL (Kj/m²): Intensidade da radiação solar que atinge a superfície horizontal. Essencial para modelos de balanço de energia.

Nota: Valores nulos à noite podem indicar ausência de radiação ou falha no sensor.

2. Termodinâmica do Ar (Temperatura e Umidade)
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): Temperatura real do ar ambiente.

TEMPERATURA DO PONTO DE ORVALHO (°C): Temperatura na qual o vapor de água presente no ar começa a se condensar (passar para o estado líquido).

TEMPERATURA MÁXIMA/MÍNIMA NA HORA ANT. (°C): Os extremos de temperatura registrados nos 60 minutos que antecederam a leitura.

TEMPERATURA ORVALHO MAX./MIN. NA HORA ANT. (°C): Extremos da temperatura de orvalho na hora anterior, úteis para medir a variação da massa de umidade.

UMIDADE RELATIVA DO AR, HORARIA (%): Relação entre a quantidade de água presente no ar e a capacidade máxima de retenção na temperatura atual.

UMIDADE REL. MAX./MIN. NA HORA ANT. (%): Oscilação da umidade relativa no período de uma hora.

3. Dinâmica Atmosférica (Pressão e Vento)
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO (mB): Força exercida pela coluna de ar no local exato do sensor (não corrigida para o nível do mar).

PRESSÃO ATMOSFERICA MAX./MIN. NA HORA ANT. (mB): Indicadores de tendência barométrica (subida ou queda da pressão), cruciais para prever frentes frias ou tempestades.

VENTO, DIREÇÃO HORARIA (gr): Direção de onde o vento sopra em graus geográficos (0° a 360°).

VENTO, VELOCIDADE HORARIA (m/s): Velocidade média do fluxo de ar no intervalo de uma hora.

VENTO, RAJADA MAXIMA (m/s): O maior valor de velocidade instantânea registrado durante a hora anterior.

In [40]:
import pandas as pd
import numpy as np
import sys
import os
import gc

import unicodedata
import re

print(f"Versão:")
print(f'Python: {sys.version}')
print(f'Pandas: {pd.__version__}')
print(f'Numpy: {np.__version__}')

Versão:
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Pandas: 2.2.2
Numpy: 2.0.2


In [41]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
path = '/content/drive/MyDrive/ML-UFF'

Mounted at /content/drive


In [42]:
def save_data_frame(df: pd.DataFrame, path_dataset: str):
  df.to_csv(path_dataset, index=True)

In [43]:
def get_data_frame(path_dataset: str):
  data = pd.read_csv(
                path_inmet,
                sep=";",
                decimal=",",
                encoding="latin1",
                skiprows=8,
            )
  return data

In [44]:
anos = [2020,2021,2022,2023,2024,2025,2026]

In [45]:
inmet_dict = {'A602':'MARAMBAIA','A621':'VILA MILITAR','A636':'JACAREPAGUA','A652':'FORTE COPACABANA'}

In [46]:
df = pd.DataFrame()
for ano in anos:
  for k,v in inmet_dict.items():
    path_inmet = f'{path}/INMET/{ano}/{k}.CSV'
    print(f'{path_inmet} - {v}')
    data = get_data_frame(path_inmet)
    data['CODIGO'] = k
    data['ESTACAO'] = v
    df = pd.concat([df, data], axis=0, ignore_index=True)

/content/drive/MyDrive/ML-UFF/INMET/2020/A602.CSV - MARAMBAIA
/content/drive/MyDrive/ML-UFF/INMET/2020/A621.CSV - VILA MILITAR
/content/drive/MyDrive/ML-UFF/INMET/2020/A636.CSV - JACAREPAGUA
/content/drive/MyDrive/ML-UFF/INMET/2020/A652.CSV - FORTE COPACABANA
/content/drive/MyDrive/ML-UFF/INMET/2021/A602.CSV - MARAMBAIA
/content/drive/MyDrive/ML-UFF/INMET/2021/A621.CSV - VILA MILITAR
/content/drive/MyDrive/ML-UFF/INMET/2021/A636.CSV - JACAREPAGUA
/content/drive/MyDrive/ML-UFF/INMET/2021/A652.CSV - FORTE COPACABANA
/content/drive/MyDrive/ML-UFF/INMET/2022/A602.CSV - MARAMBAIA
/content/drive/MyDrive/ML-UFF/INMET/2022/A621.CSV - VILA MILITAR
/content/drive/MyDrive/ML-UFF/INMET/2022/A636.CSV - JACAREPAGUA
/content/drive/MyDrive/ML-UFF/INMET/2022/A652.CSV - FORTE COPACABANA
/content/drive/MyDrive/ML-UFF/INMET/2023/A602.CSV - MARAMBAIA
/content/drive/MyDrive/ML-UFF/INMET/2023/A621.CSV - VILA MILITAR
/content/drive/MyDrive/ML-UFF/INMET/2023/A636.CSV - JACAREPAGUA
/content/drive/MyDrive/ML-UFF

In [47]:
df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
df = df.sort_values('Data')
df.set_index('Data', inplace=True)

In [48]:
df.shape, df.columns

((219072, 21),
 Index(['Hora UTC', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
        'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
        'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
        'RADIACAO GLOBAL (Kj/m²)',
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
        'TEMPERATURA DO PONTO DE ORVALHO (°C)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
        'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
        'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
        'UMIDADE RELATIVA DO AR, HORARIA (%)',
        'VENTO, DIREÇÃO HORARIA (gr) (° (gr))', 'VENTO, RAJADA MAXIMA (m/s)',
        'VENTO, VELOCIDADE HORARIA (m/s)', 'Unnamed: 19', 'CODIGO', 'ESTACAO'],
       dtype='object'))

In [49]:
df.head()

,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),...,TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19,CODIGO,ESTACAO
Data,,,,,,,,,,,,,,,,,,,,,
2020-01-01,0000 UTC,0.0,1007.1,1007.1,1006.2,NaN,24.5,19.5,26.0,24.0,...,18.7,74.0,70.0,74.0,279.0,2.9,1.2,NaN,A602,MARAMBAIA
2020-01-01,0300 UTC,0.0,1004.7,1005.0,1004.7,NaN,26.2,20.3,27.1,26.1,...,20.2,70.0,68.0,70.0,63.0,4.3,1.7,NaN,A652,FORTE COPACABANA
2020-01-01,0200 UTC,0.0,1005.0,1005.1,1004.9,NaN,26.5,20.7,26.5,23.4,...,19.3,81.0,70.0,70.0,53.0,4.0,1.6,NaN,A652,FORTE COPACABANA
2020-01-01,0100 UTC,0.0,1004.9,1004.9,1004.2,NaN,24.7,20.6,24.7,23.5,...,19.6,82.0,77.0,78.0,58.0,3.8,1.2,NaN,A652,FORTE COPACABANA
2020-01-01,0000 UTC,0.0,1004.2,1004.2,1003.9,NaN,24.0,19.7,24.3,22.8,...,19.2,82.0,77.0,77.0,80.0,3.5,1.9,NaN,A652,FORTE COPACABANA


In [50]:
df.index.min(), df.index.max()

(Timestamp('2020-01-01 00:00:00'), Timestamp('2026-03-31 00:00:00'))

In [51]:
# Retorna uma lista com os nomes das colunas
colunas_vazias = df.columns[df.isna().all()].tolist()
print(colunas_vazias)

['Unnamed: 19']


In [52]:
# Removendo colunas que todos os valores são NaNs
df.drop(columns=colunas_vazias, inplace=True)

In [53]:
# Mostra a quantidade de NaNs em cada coluna
print(df.isna().sum())

Hora UTC                                                     0
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                         13896
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     7993
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           8510
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          8510
RADIACAO GLOBAL (Kj/m²)                                  80644
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)             10816
TEMPERATURA DO PONTO DE ORVALHO (°C)                     40907
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)               11284
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)               11285
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)         41574
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)         41648
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                 41481
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 41598
UMIDADE RELATIVA DO AR, HORARIA (%)                      40875
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                   

In [54]:
# Renomenado as colunas
def limpar_nomes(coluna):
    # Converte para minúsculo
    coluna = coluna.lower()
    # Remove acentos e caracteres especiais
    coluna = unicodedata.normalize('NFKD', coluna).encode('ascii', 'ignore').decode('utf-8')
    # Substitui espaços e caracteres não alfanuméricos por underline
    coluna = re.sub(r'[^a-z0-9]+', '_', coluna)
    # Remove underlines extras no início ou fim
    return coluna.strip('_')

In [55]:
df.columns = [limpar_nomes(col) for col in df.columns]
df.columns

Index(['hora_utc', 'precipitacao_total_horario_mm',
       'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
       'pressao_atmosferica_max_na_hora_ant_aut_mb',
       'pressao_atmosferica_min_na_hora_ant_aut_mb', 'radiacao_global_kj_m2',
       'temperatura_do_ar_bulbo_seco_horaria_c',
       'temperatura_do_ponto_de_orvalho_c',
       'temperatura_maxima_na_hora_ant_aut_c',
       'temperatura_minima_na_hora_ant_aut_c',
       'temperatura_orvalho_max_na_hora_ant_aut_c',
       'temperatura_orvalho_min_na_hora_ant_aut_c',
       'umidade_rel_max_na_hora_ant_aut', 'umidade_rel_min_na_hora_ant_aut',
       'umidade_relativa_do_ar_horaria', 'vento_direcao_horaria_gr_gr',
       'vento_rajada_maxima_m_s', 'vento_velocidade_horaria_m_s', 'codigo',
       'estacao'],
      dtype='object')

In [56]:
# Retirando UTC do valor de cada linha na coluna hora_utc
df['hora_utc'] = df['hora_utc'].str.replace(' UTC', '', regex=False)

In [57]:
# Converter de string para numérico (ex: "0100" vira 100)
df['hora_utc'] = pd.to_numeric(df['hora_utc'])

# Transformar em escala de hora (0-23)
# Como o padrão do INMET é HH00, dividimos por 100
df['hora_utc'] = (df['hora_utc'] // 100).astype(int)

In [58]:
# feat enig -> sin cosine hora_utc TODO

# # Engenharia de recursos para ciclicidade da hora
# df['hora_sin'] = np.sin(2 * np.pi * df['hora_utc'] / 24)
# df['hora_cos'] = np.cos(2 * np.pi * df['hora_utc'] / 24)

In [59]:
df.head()

,hora_utc,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_m2,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s,codigo,estacao
Data,,,,,,,,,,,,,,,,,,,,
2020-01-01,0,0.0,1007.1,1007.1,1006.2,NaN,24.5,19.5,26.0,24.0,20.7,18.7,74.0,70.0,74.0,279.0,2.9,1.2,A602,MARAMBAIA
2020-01-01,3,0.0,1004.7,1005.0,1004.7,NaN,26.2,20.3,27.1,26.1,20.7,20.2,70.0,68.0,70.0,63.0,4.3,1.7,A652,FORTE COPACABANA
2020-01-01,2,0.0,1005.0,1005.1,1004.9,NaN,26.5,20.7,26.5,23.4,21.0,19.3,81.0,70.0,70.0,53.0,4.0,1.6,A652,FORTE COPACABANA
2020-01-01,1,0.0,1004.9,1004.9,1004.2,NaN,24.7,20.6,24.7,23.5,20.9,19.6,82.0,77.0,78.0,58.0,3.8,1.2,A652,FORTE COPACABANA
2020-01-01,0,0.0,1004.2,1004.2,1003.9,NaN,24.0,19.7,24.3,22.8,20.4,19.2,82.0,77.0,77.0,80.0,3.5,1.9,A652,FORTE COPACABANA


In [60]:
df.columns

Index(['hora_utc', 'precipitacao_total_horario_mm',
       'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
       'pressao_atmosferica_max_na_hora_ant_aut_mb',
       'pressao_atmosferica_min_na_hora_ant_aut_mb', 'radiacao_global_kj_m2',
       'temperatura_do_ar_bulbo_seco_horaria_c',
       'temperatura_do_ponto_de_orvalho_c',
       'temperatura_maxima_na_hora_ant_aut_c',
       'temperatura_minima_na_hora_ant_aut_c',
       'temperatura_orvalho_max_na_hora_ant_aut_c',
       'temperatura_orvalho_min_na_hora_ant_aut_c',
       'umidade_rel_max_na_hora_ant_aut', 'umidade_rel_min_na_hora_ant_aut',
       'umidade_relativa_do_ar_horaria', 'vento_direcao_horaria_gr_gr',
       'vento_rajada_maxima_m_s', 'vento_velocidade_horaria_m_s', 'codigo',
       'estacao'],
      dtype='object')

### Tratando dados faltantes

In [61]:
# Verificando a consistência dos dados após o concat
info_inmet = pd.DataFrame({
    'nulos': df.isnull().sum(),
    'percentual_nulos': (df.isnull().sum() / len(df)) * 100,
    'dtype': df.dtypes
})

print("Resumo de Dados Faltantes por Coluna:")
display(info_inmet.sort_values(by='percentual_nulos', ascending=False))

Resumo de Dados Faltantes por Coluna:


,nulos,percentual_nulos,dtype
radiacao_global_kj_m2,80644,36.811642,float64
temperatura_orvalho_min_na_hora_ant_aut_c,41648,19.011101,float64
umidade_rel_min_na_hora_ant_aut,41598,18.988278,float64
temperatura_orvalho_max_na_hora_ant_aut_c,41574,18.977323,float64
umidade_rel_max_na_hora_ant_aut,41481,18.934871,float64
temperatura_do_ponto_de_orvalho_c,40907,18.672856,float64
umidade_relativa_do_ar_horaria,40875,18.658249,float64
vento_rajada_maxima_m_s,18913,8.633235,float64
vento_velocidade_horaria_m_s,18642,8.509531,float64
vento_direcao_horaria_gr_gr,18626,8.502228,float64


In [62]:
# Tratamento por Interpolação (Dados Graduais)
# As temperaturas, pressões e umidades seguem curvas contínuas.
# A interpolação linear preenche as lacunas traçando uma linha entre os pontos conhecidos.

# Colunas que variam gradualmente
cols_graduais = [
    'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
    'pressao_atmosferica_max_na_hora_ant_aut_mb',
    'pressao_atmosferica_min_na_hora_ant_aut_mb',
    'temperatura_do_ar_bulbo_seco_horaria_c',
    'temperatura_do_ponto_de_orvalho_c',
    'temperatura_maxima_na_hora_ant_aut_c',
    'temperatura_minima_na_hora_ant_aut_c',
    'temperatura_orvalho_max_na_hora_ant_aut_c',
    'temperatura_orvalho_min_na_hora_ant_aut_c',
    'umidade_rel_max_na_hora_ant_aut',
    'umidade_rel_min_na_hora_ant_aut',
    'umidade_relativa_do_ar_horaria'
]

# Interpola buracos de até 3 horas (acima disso, o erro físico cresce muito)
df[cols_graduais] = df[cols_graduais].interpolate(method='linear', limit=24)

In [63]:
# Tratamento por Preenchimento Zero (Eventos e Radiação)
# A precipitação e a radiação têm comportamentos específicos onde o
# NaN frequentemente significa valor zero (especialmente à noite ou em períodos secos).

# Precipitação: se não há registro, assume-se que não choveu
df['precipitacao_total_horario_mm'] = df['precipitacao_total_horario_mm'].fillna(0)

# Radiação Global: preenche com 0 (assumindo falta de registro noturno)
df['radiacao_global_kj_m2'] = df['radiacao_global_kj_m2'].fillna(0)

In [64]:
# Tratamento de Vento (Propagação de Valor)
# O vento é muito instável para interpolação.
# Usamos o ffill (forward fill) para repetir o último valor conhecido por um curto período.
cols_vento = [
    'vento_direcao_horaria_gr_gr',
    'vento_rajada_maxima_m_s',
    'vento_velocidade_horaria_m_s'
]

df[cols_vento] = df[cols_vento].ffill(limit=24)

In [65]:
# Preenchimento pela Mediana (Estratégia por Estação)
cols_com_nulos = df.columns[df.isna().any()].tolist()

for col in cols_com_nulos:
    df[col] = df[col].fillna(df.groupby('codigo')[col].transform('median'))

In [66]:
# Limpeza Final (Corte de Resíduos)
# Se após esses passos ainda houver nulos, significa que a estação ficou offline por longos períodos.
# Manter esses dados "vazios" ou preenchê-los artificialmente prejudicaria seu modelo de ML.

# Remove as linhas que permaneceram com nulos nas colunas críticas
# (Essencial para garantir que o modelo não receba NaNs)
df.dropna(subset=['temperatura_do_ar_bulbo_seco_horaria_c', 'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb'], inplace=True)

# Verificação final
print(df.isna().sum())

hora_utc                                              0
precipitacao_total_horario_mm                         0
pressao_atmosferica_ao_nivel_da_estacao_horaria_mb    0
pressao_atmosferica_max_na_hora_ant_aut_mb            0
pressao_atmosferica_min_na_hora_ant_aut_mb            0
radiacao_global_kj_m2                                 0
temperatura_do_ar_bulbo_seco_horaria_c                0
temperatura_do_ponto_de_orvalho_c                     0
temperatura_maxima_na_hora_ant_aut_c                  0
temperatura_minima_na_hora_ant_aut_c                  0
temperatura_orvalho_max_na_hora_ant_aut_c             0
temperatura_orvalho_min_na_hora_ant_aut_c             0
umidade_rel_max_na_hora_ant_aut                       0
umidade_rel_min_na_hora_ant_aut                       0
umidade_relativa_do_ar_horaria                        0
vento_direcao_horaria_gr_gr                           0
vento_rajada_maxima_m_s                               0
vento_velocidade_horaria_m_s                    

In [67]:
df.shape, df.columns

((219072, 20),
 Index(['hora_utc', 'precipitacao_total_horario_mm',
        'pressao_atmosferica_ao_nivel_da_estacao_horaria_mb',
        'pressao_atmosferica_max_na_hora_ant_aut_mb',
        'pressao_atmosferica_min_na_hora_ant_aut_mb', 'radiacao_global_kj_m2',
        'temperatura_do_ar_bulbo_seco_horaria_c',
        'temperatura_do_ponto_de_orvalho_c',
        'temperatura_maxima_na_hora_ant_aut_c',
        'temperatura_minima_na_hora_ant_aut_c',
        'temperatura_orvalho_max_na_hora_ant_aut_c',
        'temperatura_orvalho_min_na_hora_ant_aut_c',
        'umidade_rel_max_na_hora_ant_aut', 'umidade_rel_min_na_hora_ant_aut',
        'umidade_relativa_do_ar_horaria', 'vento_direcao_horaria_gr_gr',
        'vento_rajada_maxima_m_s', 'vento_velocidade_horaria_m_s', 'codigo',
        'estacao'],
       dtype='object'))

In [68]:
df.index

DatetimeIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01',
               ...
               '2026-03-31', '2026-03-31', '2026-03-31', '2026-03-31',
               '2026-03-31', '2026-03-31', '2026-03-31', '2026-03-31',
               '2026-03-31', '2026-03-31'],
              dtype='datetime64[ns]', name='Data', length=219072, freq=None)

### Tratamento de Duplicidade

In [69]:
# Identificar a chave da duplicidade
# Primeiro, verifique se existem linhas onde a combinação de data (índice), hora_utc e codigo (da estação) se repete.

# Criando uma lista com as colunas que definem um registro único
chave_temporal = ['hora_utc', 'codigo']

# Contando quantas linhas estão duplicadas considerando o índice e a chave
duplicados = df.index.duplicated(keep=False) # Verifica apenas o índice 'Data'
# Ou de forma mais completa:
duplicados_completos = df.reset_index().duplicated(subset=['Data', 'hora_utc', 'codigo']).sum()

print(f"Total de registros com chave temporal repetida: {duplicados_completos}")

Total de registros com chave temporal repetida: 0


### Ordenação

In [70]:
# Garante que a coluna 'Data' (índice) seja datetime
df.index = pd.to_datetime(df.index)

In [71]:
# Ordena de forma lógica: Estação -> Data -> Hora
# Isso agrupa toda a série histórica de uma estação antes de começar a próxima
df = df.sort_values(by=['codigo', 'Data', 'hora_utc'])

In [72]:
# Exibe as primeiras linhas para conferir a cronologia por estação
display(df[['codigo', 'hora_utc', 'temperatura_do_ar_bulbo_seco_horaria_c']].head(10))

,codigo,hora_utc,temperatura_do_ar_bulbo_seco_horaria_c
Data,,,
2020-01-01,A602,0,24.5
2020-01-01,A602,1,24.6
2020-01-01,A602,2,23.5
2020-01-01,A602,3,23.5
2020-01-01,A602,4,23.4
2020-01-01,A602,5,23.2
2020-01-01,A602,6,23.1
2020-01-01,A602,7,24.3
2020-01-01,A602,8,24.2


In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 219072 entries, 2020-01-01 to 2026-03-31
Data columns (total 20 columns):
 #   Column                                              Non-Null Count   Dtype  
---  ------                                              --------------   -----  
 0   hora_utc                                            219072 non-null  int64  
 1   precipitacao_total_horario_mm                       219072 non-null  float64
 2   pressao_atmosferica_ao_nivel_da_estacao_horaria_mb  219072 non-null  float64
 3   pressao_atmosferica_max_na_hora_ant_aut_mb          219072 non-null  float64
 4   pressao_atmosferica_min_na_hora_ant_aut_mb          219072 non-null  float64
 5   radiacao_global_kj_m2                               219072 non-null  float64
 6   temperatura_do_ar_bulbo_seco_horaria_c              219072 non-null  float64
 7   temperatura_do_ponto_de_orvalho_c                   219072 non-null  float64
 8   temperatura_maxima_na_hora_ant_aut_c            

In [74]:
df.describe()

,hora_utc,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,radiacao_global_kj_m2,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
count,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000,219072.000000
mean,11.500000,0.132178,1012.774181,1013.018915,1012.524117,661.614156,23.597266,19.219957,24.199455,23.038743,19.760582,18.702967,80.861346,75.164168,78.160769,168.898030,4.533727,1.826657
std,6.922202,1.117592,4.896857,4.875696,4.911969,1027.029552,4.074087,3.158974,4.228967,3.910492,3.135841,3.195573,14.122517,16.122226,15.251463,104.303845,2.922310,1.778637
min,0.000000,0.000000,995.000000,995.300000,994.800000,0.000000,8.300000,-10.000000,8.700000,8.300000,-10.000000,-9.100000,7.000000,7.000000,7.000000,1.000000,0.000000,0.000000
25%,5.750000,0.000000,1009.300000,1009.600000,1009.100000,0.000000,21.000000,17.300000,21.450000,20.600000,17.800000,16.700000,73.000000,65.000000,69.000000,78.000000,2.400000,0.500000
50%,11.500000,0.000000,1012.500000,1012.700000,1012.200000,3.000000,23.300000,19.600000,23.800000,22.900000,20.100000,19.100000,84.000000,78.000000,81.000000,171.000000,4.000000,1.400000
75%,17.250000,0.000000,1016.000000,1016.300000,1015.800000,1095.200000,25.900000,21.500000,26.700000,25.300000,22.000000,21.033333,92.000000,88.500000,90.333333,254.000000,6.100000,2.500000
max,23.000000,97.200000,1030.900000,1031.000000,1030.600000,5356.400000,41.700000,37.400000,42.500000,41.600000,38.000000,36.000000,100.000000,100.000000,100.000000,360.000000,30.000000,18.700000


In [75]:
path_inmet_rj = '/content/drive/MyDrive/ML-UFF/inmet_rj_2020_2026.csv'

In [76]:
save_data_frame(df, path_inmet_rj)